# Day 2 · Hands-on 2 — Multiple regression and interactions
**About 30 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)


Real outcomes depend on several things at once. Adding predictors lets us estimate each effect **holding the others constant**, and interactions let an effect **differ between groups**.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to change, fix or explain *code*.
> - 🎓 **Claude / ChatGPT (other tab) = tutor you must check.** Ask it to *interpret*, then score its answer: **correct · complete · specific · actionable**.


**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for the simulated dataset, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
# Every analysis below uses ROLES (outcome, group, ...), never hard-coded column names.
# To add a dataset: write a loader function and add one entry to DATASETS.
%pip install -q lifelines
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from lifelines.datasets import load_gbsg2, load_lung

def _gbsg2():
    d = load_gbsg2()
    for c in ["tgrade", "menostat", "horTh"]:
        d[c] = d[c].astype(str)
    d["nodes4"] = (d["pnodes"] >= 4).astype(int)
    return d

def _lung():
    d = load_lung()
    d.columns = [c.replace(".", "_") for c in d.columns]
    d = d.dropna(subset=["ph_ecog"])
    d["ecog"] = d["ph_ecog"].map({0: "0", 1: "1", 2: "2+", 3: "2+"})
    d["sex"] = d["sex"].map({1: "male", 2: "female"})
    d["lost_weight"] = np.where(d["wt_loss"].isna(), np.nan, (d["wt_loss"] > 0).astype(float))
    return d

def _simulated(seed=2026, n=300):
    rng = np.random.default_rng(seed)
    genotype = rng.choice(["AA", "AB", "BB"], size=n, p=[0.4, 0.4, 0.2])
    sex = rng.choice(["F", "M"], size=n)
    age = rng.uniform(30, 80, n) + np.where(genotype == "BB", 8, 0)          # BB carriers are older
    g_eff = pd.Series({"AA": 0, "AB": 3, "BB": 8})[genotype].to_numpy()
    slope = np.where(sex == "M", 0.7, 0.4)                                    # interaction: steeper in men
    biomarker = 50 + slope * (age - 50) + g_eff + rng.normal(0, 6, n)
    logit = -1 + 0.05 * (age - 50) + pd.Series({"AA": 0, "AB": 0.4, "BB": 1.0})[genotype].to_numpy()
    disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    mu = np.exp(0.5 + 0.02 * (age - 50) + pd.Series({"AA": 0, "AB": 0.3, "BB": 0.7})[genotype].to_numpy())
    alpha = 0.8
    lesions = rng.negative_binomial(1 / alpha, 1 / (1 + alpha * mu))
    return pd.DataFrame(dict(biomarker=biomarker.round(1), genotype=genotype, sex=sex, age=age.round(1),
                             disease=disease, lesions=lesions))

def _my_data():
    # Upload a CSV to Colab (folder icon on the left) and edit the roles in DATASETS["my_data"]
    return pd.read_csv("my_data.csv")

DATASETS = {
    "gbsg2": dict(load=_gbsg2,
        about="GBSG2 breast cancer trial: 686 patients",
        outcome="tsize", outcome_label="Tumour size (mm)",
        group="tgrade", group_label="Tumour grade",
        x="pnodes", x_label="Positive lymph nodes",
        group2="menostat", group2_label="Menopausal status",
        binary="nodes4", binary_label="4 or more positive lymph nodes",
        binary_x="tsize", binary_group="tgrade",
        count="pnodes", count_label="Positive lymph nodes", count_x="tsize", count_group="tgrade"),
    "lung": dict(load=_lung,
        about="NCCTG advanced lung cancer study: 227 patients",
        outcome="pat_karno", outcome_label="Patient-rated Karnofsky score",
        group="ecog", group_label="ECOG performance status",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="lost_weight", binary_label="Lost weight in last 6 months",
        binary_x="age", binary_group="ecog",
        count=None),
    "simulated": dict(load=_simulated,
        about="Simulated biomarker study: 300 people, true effects known",
        outcome="biomarker", outcome_label="Biomarker level",
        group="genotype", group_label="Genotype",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="disease", binary_label="Disease present",
        binary_x="age", binary_group="genotype",
        count="lesions", count_label="Number of lesions", count_x="age", count_group="genotype",
        truth={
            "group": "AB vs AA = +3, BB vs AA = +8 (adjusted for age). BB carriers are also ~8 years older.",
            "x": "age slope = 0.4 per year in women, 0.7 in men",
            "interaction": "difference in age slopes (M vs F) = +0.3 per year",
            "binary": "odds ratio per year of age = 1.05; AB vs AA = 1.49; BB vs AA = 2.72",
            "count": "rate ratio per year of age = 1.02; AB vs AA = 1.35; BB vs AA = 2.01; overdispersed (alpha = 0.8)",
        }),
    "my_data": dict(load=_my_data,
        about="Your own CSV: edit these roles to match your column names",
        outcome="my_outcome", outcome_label="My outcome",
        group="my_group", group_label="My group",
        x="my_numeric_predictor", x_label="My numeric predictor",
        group2="my_two_level_group", group2_label="My two-level group",
        binary="my_yes_no_outcome", binary_label="My yes/no outcome (coded 0/1)",
        binary_x="my_numeric_predictor", binary_group="my_group",
        count=None),
}

def load_dataset(name):
    R = dict(DATASETS[name])
    d = R.pop("load")()
    for role in ["group", "group2", "binary_group", "count_group"]:
        if R.get(role):
            d[R[role]] = d[R[role]].astype(str)
    R["levels"] = sorted(d[R["group"]].dropna().unique())
    R.setdefault("truth", {})
    print(f"✅ {R['about']}  ({len(d)} rows)")
    print(f"   outcome: {R['outcome']} ({R['outcome_label']})   group: {R['group']} {R['levels']}")
    print(f"   numeric predictor: {R['x']}   second group: {R['group2']}")
    return d, R

def cat(var, ref):
    """Categorical term with an explicit reference level."""
    return f"C({var}, Treatment(reference='{ref}'))"

def nice(term):
    """Turn statsmodels term names into readable labels."""
    m = re.fullmatch(r"C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\](.*)", term)
    if m:
        var, ref, lvl, rest = m.groups()
        label = f"{var}: {lvl} vs {ref}"
        return label + (f" × {rest.lstrip(':')}" if rest else "")
    m = re.fullmatch(r"(\w+):C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\]", term)
    if m:
        x, var, ref, lvl = m.groups()
        return f"{x} × {var}: {lvl} vs {ref}"
    return term

def tidy(fit, exponentiate=False):
    """Coefficient table: estimate, 95% CI, p-value."""
    ci = fit.conf_int()
    t = pd.DataFrame({"estimate": fit.params, "CI low": ci[0], "CI high": ci[1], "p-value": fit.pvalues})
    t = t.drop(index=[i for i in t.index if i == "alpha"], errors="ignore")
    if exponentiate:
        t[["estimate", "CI low", "CI high"]] = np.exp(t[["estimate", "CI low", "CI high"]])
    t.index = [nice(i) for i in t.index]
    return t.round({"estimate": 3, "CI low": 3, "CI high": 3}).assign(**{"p-value": t["p-value"].map(lambda p: f"{p:.2g}")})

def show_truth(R, key):
    if key in R["truth"]:
        print(f"🎯 Truth used to simulate: {R['truth'][key]}")

print("✅ Setup complete")

## Choose a dataset
Everything in this notebook is written in terms of **roles** (an outcome, a group, a numeric predictor), so the same cells work on any dataset. Start with the one your facilitator names, then try `simulated`: it was generated with known effects, so you can check whether the model recovers the truth.

In [ ]:
# 🔧 CHANGE THIS: which dataset to use (all cells below adapt automatically)
dataset = "gbsg2"  # @param ["gbsg2", "lung", "simulated", "my_data"]

df, R = load_dataset(dataset)
df.head()

## A. Adjusting changes the answer
### 🧭 Concept
The group effect *without* adjustment mixes the group's own effect with anything else that differs between groups. Adding a predictor separates them.

✍️ **Predict first:** when you adjust for the numeric predictor, will the group coefficients grow, shrink or stay the same?

In [ ]:
# 🔧 CHANGE THIS: variable to adjust for (any numeric column)
adjust_for = R["x"]

y, g, ref = R["outcome"], R["group"], R["levels"][0]
d = df.dropna(subset=[y, g, adjust_for])         # same patients in both models
crude = smf.ols(f"{y} ~ {cat(g, ref)}", data=d).fit()
adjusted = smf.ols(f"{y} ~ {cat(g, ref)} + {adjust_for}", data=d).fit()

compare = pd.DataFrame({"crude": tidy(crude)["estimate"], "adjusted": tidy(adjusted)["estimate"]})
display(compare)
print(f"\nWhy did it change? Mean {adjust_for} in each group:")
print(d.groupby(g)[adjust_for].mean().round(2).to_dict())
show_truth(R, "group")

### ✅ Concept check
Fill in the sentence: *The group coefficient changed after adjustment because {adjust_for} differs between groups **and** ...*

<details><summary>Show answer</summary>

... it is itself associated with the outcome. A variable changes a coefficient when it is related to **both** the predictor and the outcome. That is the definition of a confounder (if it isn't a consequence of the predictor). Adjustment only helps for confounders you measured.
</details>

## B. Interactions: does the slope differ between groups?
### 🧭 Concept
- **Additive model** `y ~ x + group2`: one slope, shifted between groups (parallel lines).
- **Interaction model** `y ~ x * group2`: each group gets its own slope. The interaction coefficient is the **difference in slopes**.

Switch between the two and compare the plots.

In [ ]:
# 🔧 CHANGE THIS: "additive" or "interaction"
model = "additive"

y, x, g2 = R["outcome"], R["x"], R["group2"]
d = df.dropna(subset=[y, x, g2])
lv = sorted(d[g2].unique())
op = "+" if model == "additive" else "*"
fit = smf.ols(f"{y} ~ {x} {op} {cat(g2, lv[0])}", data=d).fit()

plt.figure(figsize=(6, 4))
xs = np.linspace(d[x].min(), d[x].max(), 50)
for level in lv:
    sub = d[d[g2] == level]
    plt.scatter(sub[x], sub[y], alpha=0.3, s=12, label=level)
    pred = fit.predict(pd.DataFrame({x: xs, g2: level}))
    plt.plot(xs, pred, linewidth=2.5)
plt.xlabel(R["x_label"]); plt.ylabel(R["outcome_label"]); plt.legend(title=R["group2_label"])
plt.title(f"{model} model"); plt.show()

display(tidy(fit))
for level in lv:
    slope = fit.predict(pd.DataFrame({x: [1], g2: level}))[0] - fit.predict(pd.DataFrame({x: [0], g2: level}))[0]
    print(f"Slope of {x} for {level}: {slope:.3f}")
show_truth(R, "interaction")

### ✅ Concept check
1. In the interaction model, what does the `x × group2` coefficient equal? Check it against the two printed slopes.
2. The interaction p-value is large. Does that show the slopes are equal?

<details><summary>Show answers</summary>

1. The difference between the two slopes.
2. No. It means the data can't distinguish them. Look at the confidence interval: interactions need much larger samples than main effects to estimate precisely.
</details>

## 🤖 AI prompts
**🎓 Claude / ChatGPT (tutor) — a known failure point: causal language.** The cell below prints your adjusted model from section A as a prompt. Paste it.

In [ ]:
# Generates a prompt from YOUR adjusted model. Nothing to change.
print(f"""I fitted a linear regression of {R['outcome_label']} on {R['group_label']} and {R['x_label']} in observational data. Results:
{tidy(adjusted).to_string()}
Write two sentences interpreting these results for the results section of a paper.""")

Score with the rubric. Watch for *causes*, *increases*, *leads to*, *effect of*, or "raising X by 1 would...". With observational data a good answer says **associated with** and **holding the other variables constant**.

**🧮 Gemini in Colab (calculator):** select the cell in section B and ask:
> *Add the group variable from section A to this model as well.*

Check: did the formula change the way you expected? Did the slopes move?

## 💬 Discuss with your partner
- Name a variable in your own research that would be a confounder, and one that would be a **mediator** (on the path from cause to outcome). Should you adjust for both?
- When would you include an interaction in a model *before* seeing the data?
- How would you explain "holding the others constant" to a colleague in one sentence?